In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Convert timestamp columns to UTC datetimes (Unix seconds -> UTC)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Aggregations for metric.csv
metric_agg = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
    .sort_values('count', ascending=False)
)
metric_agg_top50 = metric_agg.head(50)

# Aggregations for trace.csv
trace_agg = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
    .sort_values('count', ascending=False)
)
trace_agg_top50 = trace_agg.head(50)

# Aggregations for log.csv
log_agg = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
    .sort_values('count', ascending=False)
)
log_agg_top50 = log_agg.head(50)

# Aggregations for error_logs.csv: per-cmdb_id counts and min/max timestamps
error_agg = (
    error_df
    .groupby('cmdb_id', as_index=False)
    .agg(
        count=('message', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max')
    )
    .sort_values('count', ascending=False)
)

# Up to top 20 sample messages sorted by timestamp ascending
error_samples = error_df.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(20)

# Overall min and max timestamps present in each file
ranges = []
for name, df in [('metric.csv', metric_df), ('trace.csv', trace_df), ('log.csv', log_df), ('error_logs.csv', error_df)]:
    if df.shape[0] == 0:
        ranges.append({'file': name, 'min_timestamp': pd.NaT, 'max_timestamp': pd.NaT})
    else:
        ranges.append({'file': name, 'min_timestamp': df['timestamp'].min(), 'max_timestamp': df['timestamp'].max()})
ranges_df = pd.DataFrame(ranges)

# Compact outputs (limited rows as requested)
metric_agg_top50, trace_agg_top50, log_agg_top50, error_agg, error_samples, ranges_df

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregations (timestamps in UTC):\n"
    "- metric.csv: data range 2024-01-23 06:53:00+00:00 to 2024-01-23 07:17:00+00:00. "
    "Up to 50 (cmdb_id, kpi_name) groups returned, top groups have count=25. "
    "Examples: ts-voucher-service/socket (count=25, P95=4.0), "
    "ts-admin-basic-info-service/cpu (count=25, P95≈0.5305), "
    "ts-train-service/mem (count=25, P95≈254,437,321.39).\n"
    "- trace.csv: data range 2024-01-23 06:53:00+00:00 to 2024-01-23 07:17:00+00:00. "
    "Up to 50 (cmdb_id, trace_name) groups returned, counts ~22–23. "
    "Top example: ts-price-service/trace.to_ts-price-service.row_count (count=23, P95=307.4).\n"
    "- log.csv: data range 2024-01-23 06:53:00+00:00 to 2024-01-23 07:17:00+00:00. "
    "Up to 50 (cmdb_id, log_name) groups returned, counts mostly 19–20. "
    "Examples: ts-admin-basic-info-service/log.row_count (count=20, P95=31.2); many services have log.error_count with P95=0.\n"
    "- error_logs.csv: no records present (per-cmdb_id aggregation empty); file min/max timestamps are NaT (no data).\n"
    "Overall: metric, trace, and log telemetry cover the same 24-minute window; error logs are empty for this period."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count             min_timestamp             max_timestamp        p95_value
374           ts-voucher-service      socket     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         4.000000
0    ts-admin-basic-info-service         cpu     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.530469
1    ts-admin-basic-info-service      diskio     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00     58069.683878
2    ts-admin-basic-info-service  latency-50     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.016796
3    ts-admin-basic-info-service  latency-90     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.023359
..                           ...         ...    ...                       ...                       ...              ...
325            ts-travel-service      diskio     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00     60007.766282
326            ts-travel-service       error     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.000000
311             ts-train-service  latency-50     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.006947
312             ts-train-service  latency-90     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00         0.009519
313             ts-train-service         mem     25 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00 254437321.386667

[50 rows x 6 columns],                 cmdb_id                                  trace_name  count             min_timestamp             max_timestamp  p95_value
563    ts-price-service         trace.to_ts-price-service.row_count     23 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00 307.400000
558    ts-price-service      trace.from_ts-price-service.error_rate     23 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.000000
560    ts-price-service     trace.to_ts-price-service.duration_mean     23 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.001710
561    ts-price-service      trace.to_ts-price-service.duration_p95     23 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.002320
562    ts-price-service        trace.to_ts-price-service.error_rate     23 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.000000
..                  ...                                         ...    ...                       ...                       ...        ...
167     ts-auth-service                   trace.from_root.row_count     22 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  87.800000
7                  root  trace.to_ts-admin-travel-service.row_count     22 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  11.000000
93                 root     trace.to_ts-travel-service.duration_p95     22 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   2.007641
803  ts-travel2-service                   trace.from_root.row_count     22 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  80.850000
95                 root        trace.to_ts-travel-service.row_count     22 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  24.900000

[50 rows x 6 columns],                         cmdb_id         log_name  count             min_timestamp             max_timestamp  p95_value
0   ts-admin-basic-info-service  log.error_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.000000
1   ts-admin-basic-info-service    log.row_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  31.200000
2       ts-admin-travel-service  log.error_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00   0.000000
3       ts-admin-travel-service    log.row_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  33.000000
7          ts-assurance-service    log.row_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00  16.000000
..                          ...              ...    ...                       ...                       ...        ...
18       ts-consign-price-mongo  log.error_count     19 2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00   0.000000
14              ts-config-mongo  log.error_count     19 2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00   0.000000
15              ts-config-mongo    log.row_count     19 2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00   1.000000
16             ts-consign-mongo  log.error_count     19 2024-01-23 06:53:00+00:00 2024-01-23 07:16:00+00:00   0.000000
17             ts-consign-mongo    log.row_count     19 2024-01-23 06:53:00+00:00 2024-01-23 07:16:00+00:00   1.000000

[50 rows x 6 columns], Empty DataFrame
Columns: [cmdb_id, count, min_timestamp, max_timestamp]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],              file             min_timestamp             max_timestamp
0      metric.csv 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00
1       trace.csv 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00
2         log.csv 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00
3  error_logs.csv                       NaT                       NaT)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing metric_df loaded earlier; ensure timestamp is datetime UTC
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (before filtering)
global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)
metric_window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# total records in incident window
total_records_in_window = int(metric_window_df.shape[0])

# 3) Attach global_p95 to windowed records
metric_window_df = metric_window_df.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Determine anomalies per row:
# - if global_p95 == 0 => anomaly if value > 0
# - else anomaly if value >= global_p95
metric_window_df['is_anomaly'] = np.where(
    metric_window_df['global_p95'] == 0,
    metric_window_df['value'] > 0,
    metric_window_df['value'] >= metric_window_df['global_p95']
)

# 5) Aggregate anomaly summaries per group (only groups present in the window)
# Anomaly aggregates (counts and earliest/latest anomaly timestamps)
anoms_df = (
    metric_window_df[metric_window_df['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# Max value in the window per group (needed for severity ratio)
max_in_window_df = (
    metric_window_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Merge aggregates with global_p95
summary_df = (
    anoms_df
    .merge(max_in_window_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
)

# 6) Compute severity_ratio = max_value_in_window / global_p95 (use None when global_p95 == 0)
def compute_severity(row):
    gp = row['global_p95']
    if pd.isna(gp) or gp == 0:
        return None
    return row['max_value_in_window'] / gp

summary_df['severity_ratio'] = summary_df.apply(compute_severity, axis=1)

# 7) Final ordering and limit to top 20 by anomaly_count desc
summary_top20 = (
    summary_df
    .sort_values(by='anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count',
             'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
             'max_value_in_window', 'global_p95', 'severity_ratio']]
    .head(20)
)

# 8) Number of distinct (cmdb_id,kpi_name) that had >=1 anomaly
distinct_groups_with_anomaly = int(summary_df.shape[0])

# Return compact results
summary_top20, total_records_in_window, distinct_groups_with_anomaly

```
Out[2]:
```
summary = (
    f"Summary of anomaly analysis for metric.csv (global P95 computed from full series):\n"
    f"- Incident window: 2024-01-23 06:50:39 UTC to 2024-01-23 07:20:39 UTC (inclusive).\n"
    f"- Total metric records in window: {total_records_in_window}.\n"
    f"- Number of distinct (cmdb_id, kpi_name) with ≥1 anomaly: {distinct_groups_with_anomaly}.\n"
    f"- Top anomaly count observed: {int(summary_df['anomaly_count'].max())} occurrences; "
    f"{int((summary_df['anomaly_count'] == summary_df['anomaly_count'].max()).sum())} groups reached this count.\n"
    f"- Sample top groups (cmdb_id/kpi_name[count]): "
    + ", ".join(
        f"{r['cmdb_id']}/{r['kpi_name']}[{int(r['anomaly_count'])}]"
        for _, r in summary_top20.head(5).iterrows()
    )
    + ".\n"
    f"- Earliest anomaly (in top sample): {summary_top20['earliest_anomaly_timestamp'].min()} UTC; "
    f"Latest anomaly (in top sample): {summary_top20['latest_anomaly_timestamp'].max()} UTC.\n"
    f"- Notable patterns: many 'socket' metrics across multiple mongo services and 'mem' for mysql services appear in the top anomalies. "
    f"Severity ratios in the top results are ~1.0, meaning the max value in the window equals the global P95 for those groups.\n"
    f"- Anomaly rule used: value >= global P95 (or value > 0 when global P95 == 0).\n"
    f"Overall conclusion: the incident window contains many metric groups hitting their global P95 (most commonly socket metrics on mongo services and mem on mysql), "
    f"with {int((summary_df['anomaly_count'] > 0).sum())} total anomalous groups detected (of which {distinct_groups_with_anomaly} have ≥1 anomaly in the window)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window       global_p95  severity_ratio
126            ts-food-mysql      mem             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00     599863296.000000 599863296.000000        1.000000
333            ts-user-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000
40             ts-auth-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             6.000000         6.000000        1.000000
127            ts-food-mysql   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             2.000000         2.000000        1.000000
113        ts-food-map-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             5.000000         5.000000        1.000000
160           ts-order-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             6.000000         6.000000        1.000000
275   ts-ticket-office-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             2.000000         2.000000        1.000000
80    ts-consign-price-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000
182         ts-payment-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000
76          ts-consign-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000
289           ts-train-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             6.000000         6.000000        1.000000
152    ts-notification-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             2.000000         2.000000        1.000000
347         ts-voucher-mysql   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             2.000000         2.000000        1.000000
346         ts-voucher-mysql      mem             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00     600387584.000000 600387584.000000        1.000000
300          ts-travel-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             5.000000         5.000000        1.000000
216           ts-price-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             6.000000         6.000000        1.000000
148          ts-news-service   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             1.000000         1.000000        1.000000
124            ts-food-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000
65           ts-config-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             6.000000         6.000000        1.000000
138  ts-inside-payment-mongo   socket             25  2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00             4.000000         4.000000        1.000000, 9375, 351)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df from previous run; ensure timestamp is parsed as UTC datetimes per rule 10
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True)

# Compute global P95 per (cmdb_id, kpi_name) using the full series (must be done BEFORE filtering)
# Reuse global_p95_df if exists, else compute
try:
    global_p95_df
except NameError:
    global_p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

# Restrict to the five services
services = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]

# Filter metric records to the incident window and the five services
metric_window_five = metric_df[
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    (metric_df['cmdb_id'].isin(services))
].copy()

# Total metric records in-window for these five services
total_records_window_five = int(metric_window_five.shape[0])

# Attach global_p95 to these windowed records
metric_window_five = metric_window_five.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalies per row:
# - if global_p95 == 0 => anomaly if value > 0
# - else anomaly if value >= global_p95
metric_window_five['is_anomaly'] = np.where(
    metric_window_five['global_p95'] == 0,
    metric_window_five['value'] > 0,
    metric_window_five['value'] >= metric_window_five['global_p95']
)

# Aggregate anomaly summaries per group (only groups with anomalies)
anoms_five = (
    metric_window_five[metric_window_five['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# Max value in the window per group (needed for severity ratio), computed over window (even if no anomaly)
max_in_window_five = (
    metric_window_five
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Merge anomaly aggregates with max_in_window and global_p95
summary_five = (
    anoms_five
    .merge(max_in_window_five, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute severity_ratio = max_value_in_window / global_p95 (use None when global_p95 == 0 or NaN)
def severity(row):
    gp = row['global_p95']
    if pd.isna(gp) or gp == 0:
        return None
    return row['max_value_in_window'] / gp

summary_five['severity_ratio'] = summary_five.apply(severity, axis=1)

# Final ordering and limit to top 20 by anomaly_count desc
summary_top20 = (
    summary_five
    .sort_values(by='anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count',
             'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
             'max_value_in_window', 'global_p95', 'severity_ratio']]
    .head(20)
)

# Number of distinct (cmdb_id, kpi_name) within these services that have >=1 anomaly
distinct_groups_with_anomaly_five = int(summary_five.shape[0])

# Return compact results
summary_top20, total_records_window_five, distinct_groups_with_anomaly_five

```
Out[3]:
```
```python
summary = (
    "Incident window: 2024-01-23 06:50:39 UTC — 2024-01-23 07:20:39 UTC.\n"
    "Services analyzed: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.\n"
    f"Total metric records in-window for these services: 975.\n"
    f"Number of distinct (cmdb_id, kpi_name) with ≥1 anomaly: 35.\n"
    "Anomaly rule: value >= global P95 (for groups with global P95==0, value>0 counted as anomaly).\n"
    "Key observations:\n"
    "- Most anomalous groups saw anomaly_count = 2 occurrences during the window.\n"
    "- Examples of top anomalous groups (all anomaly_count=2):\n"
    "  * ts-auth-service: cpu, diskio, latency-50, latency-90, mem, socket, workload\n"
    "  * ts-order-service: cpu, diskio, latency-50, latency-90, mem, socket, workload\n"
    "  * ts-route-service: cpu, diskio, latency-50, latency-90, mem, socket\n"
    "- Notable high-severity spikes:\n"
    "  * ts-route-service / diskio — severity_ratio ≈ 3.77 (max_in_window ~1,062,579 vs global P95 ~281,534)\n"
    "  * ts-order-service / latency-90 — severity_ratio ≈ 2.55 (max_in_window ~0.1816 vs global P95 ~0.0713)\n"
    "Interpretation: multiple KPIs across auth, order, and route services exceeded their historical P95 during the incident window; "
    "the most striking deviations are a large disk I/O spike on ts-route-service and a latency spike on ts-order-service. "
    "These are likely the highest-priority anomalies to investigate further."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window       global_p95  severity_ratio
0    ts-auth-service         cpu              2  2024-01-23 06:54:00+00:00 2024-01-23 07:14:00+00:00            15.164321        14.385320        1.054152
1    ts-auth-service      diskio              2  2024-01-23 07:09:00+00:00 2024-01-23 07:17:00+00:00         61889.887676     60876.670230        1.016644
2    ts-auth-service  latency-50              2  2024-01-23 06:57:00+00:00 2024-01-23 07:14:00+00:00             0.255296         0.250672        1.018448
3    ts-auth-service  latency-90              2  2024-01-23 07:12:00+00:00 2024-01-23 07:14:00+00:00             0.731843         0.721328        1.014578
4    ts-auth-service         mem              2  2024-01-23 07:08:00+00:00 2024-01-23 07:12:00+00:00     245383031.466667 245382922.240000        1.000000
5    ts-auth-service      socket              2  2024-01-23 07:00:00+00:00 2024-01-23 07:12:00+00:00            13.966667        13.716667        1.018226
6    ts-auth-service    workload              2  2024-01-23 06:54:00+00:00 2024-01-23 07:00:00+00:00             2.883350         2.827250        1.019843
7   ts-order-service         cpu              2  2024-01-23 06:53:00+00:00 2024-01-23 07:02:00+00:00             4.152737         4.122099        1.007432
8   ts-order-service      diskio              2  2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00         63612.013176     61578.921066        1.033016
9   ts-order-service  latency-50              2  2024-01-23 07:02:00+00:00 2024-01-23 07:06:00+00:00             0.013243         0.011833        1.119136
10  ts-order-service  latency-90              2  2024-01-23 06:55:00+00:00 2024-01-23 07:00:00+00:00             0.181602         0.071276        2.547879
11  ts-order-service         mem              2  2024-01-23 07:16:00+00:00 2024-01-23 07:17:00+00:00     310746521.600000 310470915.413333        1.000888
12  ts-order-service      socket              2  2024-01-23 07:00:00+00:00 2024-01-23 07:07:00+00:00            14.616667        14.540000        1.005273
13  ts-order-service    workload              2  2024-01-23 06:53:00+00:00 2024-01-23 07:02:00+00:00             7.365048         6.482187        1.136198
14  ts-route-service         cpu              2  2024-01-23 06:53:00+00:00 2024-01-23 06:54:00+00:00             6.051851         5.870794        1.030840
15  ts-route-service      diskio              2  2024-01-23 07:05:00+00:00 2024-01-23 07:06:00+00:00       1062579.069677    281534.395389        3.774242
16  ts-route-service  latency-50              2  2024-01-23 07:06:00+00:00 2024-01-23 07:08:00+00:00             0.005649         0.005336        1.058623
17  ts-route-service  latency-90              2  2024-01-23 07:07:00+00:00 2024-01-23 07:13:00+00:00             0.140193         0.136154        1.029664
18  ts-route-service         mem              2  2024-01-23 07:12:00+00:00 2024-01-23 07:15:00+00:00     266009736.533333 265947839.146667        1.000233
19  ts-route-service      socket              2  2024-01-23 07:06:00+00:00 2024-01-23 07:17:00+00:00            18.583333        18.213333        1.020315, 975, 35)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Ensure trace_df timestamp is parsed as UTC datetimes (reuse existing variable)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True)

# Candidate services
services = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]

# Compute global P95 per (cmdb_id, trace_name) using the full series BEFORE filtering
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

# Build mask for trace rows related to candidate services:
# - rows where cmdb_id is one of the services
# - OR trace_name mentions to_ts-<service_suffix> or from_ts-<service_suffix>
suffixes = [s.replace('ts-', '', 1) for s in services]  # e.g., "auth-service"
# Initialize boolean mask
mask_related = trace_df['cmdb_id'].isin(services)

# Expand mask by checking trace_name substrings for each service suffix
for suf in suffixes:
    mask_related |= trace_df['trace_name'].str.contains(f'to_ts-{suf}', na=False)
    mask_related |= trace_df['trace_name'].str.contains(f'from_ts-{suf}', na=False)

# Filter trace records to the incident window and related traces
trace_window_related = trace_df[
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end) &
    (mask_related)
].copy()

# Total trace records in-window for candidate-related traces
total_records_window_trace_related = int(trace_window_related.shape[0])

# Attach global_p95 to windowed records
trace_window_related = trace_window_related.merge(
    global_p95_trace, on=['cmdb_id', 'trace_name'], how='left'
)

# Determine anomalies per row:
# - if global_p95 == 0 => anomaly if value > 0
# - else anomaly if value >= global_p95
trace_window_related['is_anomaly'] = np.where(
    trace_window_related['global_p95'] == 0,
    trace_window_related['value'] > 0,
    trace_window_related['value'] >= trace_window_related['global_p95']
)

# Aggregate anomaly summaries per group (only groups with anomalies)
anoms_trace = (
    trace_window_related[trace_window_related['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# Max value in the window per group (computed over the window)
max_in_window_trace = (
    trace_window_related
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Merge anomaly aggregates with max_in_window and global_p95
summary_trace = (
    anoms_trace
    .merge(max_in_window_trace, on=['cmdb_id', 'trace_name'], how='left')
    .merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')
)

# Compute severity_ratio = max_value_in_window / global_p95 (use None when global_p95 == 0 or NaN)
def compute_sev(r):
    gp = r['global_p95']
    if pd.isna(gp) or gp == 0:
        return None
    return r['max_value_in_window'] / gp

summary_trace['severity_ratio'] = summary_trace.apply(compute_sev, axis=1)

# Final ordering and limit to top 20 by anomaly_count desc
summary_top20 = (
    summary_trace
    .sort_values(by='anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count',
             'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
             'max_value_in_window', 'global_p95', 'severity_ratio']]
    .head(20)
)

# Number of distinct (cmdb_id, trace_name) groups with >=1 anomaly
distinct_groups_with_anomaly_trace = int(summary_trace.shape[0])

# Return compact results
summary_top20, total_records_window_trace_related, distinct_groups_with_anomaly_trace

```
Out[4]:
```
```python
summary = (
    "Trace anomaly summary (timestamps UTC):\n"
    "- Incident window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39 UTC.\n"
    f"- Total trace records in-window related to the candidate services: 4240.\n"
    f"- Number of distinct (cmdb_id, trace_name) groups with ≥1 anomaly: 168.\n\n"
    "Key findings:\n"
    "- The highest anomaly counts observed: 4 occurrences for\n"
    "  * ts-admin-travel-service / trace.to_ts-travel-service.row_count\n"
    "  * ts-travel-service / trace.from_ts-admin-travel-service.row_count\n"
    "- Many groups (the majority of top results) show anomaly_count = 2 during the window.\n"
    "- Notable services with repeated trace anomalies: ts-auth-service, ts-order-service, ts-travel-service, and related 'root' traces that represent cross-service calls.\n\n"
    "Notable severity spikes (max / global P95 > 1):\n"
    "- ts-auth-service: trace.to_ts-auth-service.duration_p95 — severity ≈ 1.45 (clear latency spike)\n"
    "- ts-auth-service: several duration_mean/duration_p95 and row_count traces show small but consistent exceedances (severity ≈ 1.00–1.45).\n"
    "- ts-order-service: trace.to_ts-order-service.* traces show meaningful exceedances (severity ≈ 1.03–1.11).\n"
    "- ts-travel-service and ts-admin-travel-service: row_count/duration traces exceeded P95 (severity ≈ 1.08–1.17).\n\n"
    "Interpretation / next steps:\n"
    "- Multiple traces indicate elevated latency/row_count between services, especially around auth, order, and travel flows.\n"
    "- The top candidates to investigate first are the auth-related duration_p95 spike (highest severity) and the travel-admin row_count anomalies (highest counts).\n"
    "- Also review cross-service 'root' traces (to_ts- and from_ts- entries) for end-to-end request bottlenecks affecting these services."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                    trace_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio
17   ts-admin-travel-service          trace.to_ts-travel-service.row_count              4  2024-01-23 06:54:00+00:00 2024-01-23 07:03:00+00:00            11.000000   11.000000        1.000000
131        ts-travel-service  trace.from_ts-admin-travel-service.row_count              4  2024-01-23 06:54:00+00:00 2024-01-23 07:03:00+00:00            11.000000   11.000000        1.000000
2                       root            trace.to_ts-auth-service.row_count              2  2024-01-23 06:54:00+00:00 2024-01-23 07:00:00+00:00            88.000000   87.800000        1.002278
0                       root        trace.to_ts-auth-service.duration_mean              2  2024-01-23 06:56:00+00:00 2024-01-23 07:05:00+00:00             0.331861    0.316451        1.048697
14                      root          trace.to_ts-travel-service.row_count              2  2024-01-23 07:02:00+00:00 2024-01-23 07:05:00+00:00            27.000000   24.900000        1.084337
12                      root      trace.to_ts-travel-service.duration_mean              2  2024-01-23 07:06:00+00:00 2024-01-23 07:08:00+00:00             1.177968    1.077296        1.093449
13                      root       trace.to_ts-travel-service.duration_p95              2  2024-01-23 07:09:00+00:00 2024-01-23 07:16:00+00:00             2.343556    2.007641        1.167318
1                       root         trace.to_ts-auth-service.duration_p95              2  2024-01-23 06:55:00+00:00 2024-01-23 06:59:00+00:00             0.820397    0.725284        1.131139
23           ts-auth-service          trace.from_ts-auth-service.row_count              2  2024-01-23 06:54:00+00:00 2024-01-23 07:00:00+00:00           350.000000  350.000000        1.000000
21           ts-auth-service      trace.from_ts-auth-service.duration_mean              2  2024-01-23 07:01:00+00:00 2024-01-23 07:02:00+00:00             0.003715    0.003691        1.006483
25           ts-auth-service         trace.to_ts-auth-service.duration_p95              2  2024-01-23 07:01:00+00:00 2024-01-23 07:17:00+00:00             0.006918    0.004762        1.452677
24           ts-auth-service        trace.to_ts-auth-service.duration_mean              2  2024-01-23 07:01:00+00:00 2024-01-23 07:02:00+00:00             0.003715    0.003691        1.006483
26           ts-auth-service            trace.to_ts-auth-service.row_count              2  2024-01-23 06:54:00+00:00 2024-01-23 07:00:00+00:00           350.000000  350.000000        1.000000
19           ts-auth-service                  trace.from_root.duration_p95              2  2024-01-23 06:55:00+00:00 2024-01-23 06:59:00+00:00             0.820397    0.725284        1.131139
18           ts-auth-service                 trace.from_root.duration_mean              2  2024-01-23 06:56:00+00:00 2024-01-23 07:05:00+00:00             0.331861    0.316451        1.048697
22           ts-auth-service       trace.from_ts-auth-service.duration_p95              2  2024-01-23 07:01:00+00:00 2024-01-23 07:17:00+00:00             0.006918    0.004762        1.452677
61          ts-order-service        trace.to_ts-order-service.duration_p95              2  2024-01-23 06:59:00+00:00 2024-01-23 07:07:00+00:00             0.035736    0.033149        1.078042
127        ts-travel-service                  trace.from_root.duration_p95              2  2024-01-23 07:09:00+00:00 2024-01-23 07:16:00+00:00             2.343556    2.007641        1.167318
60          ts-order-service       trace.to_ts-order-service.duration_mean              2  2024-01-23 06:59:00+00:00 2024-01-23 07:06:00+00:00             0.067452    0.060778        1.109811
62          ts-order-service           trace.to_ts-order-service.row_count              2  2024-01-23 07:02:00+00:00 2024-01-23 07:05:00+00:00           504.000000  487.000000        1.034908, 4240, 168)```
```

In [ ]:
# In[5]:

import re
import numpy as np
import pandas as pd

# ensure trace_df and timestamp are available and parsed as UTC
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (before filtering)
try:
    global_p95_trace
except NameError:
    global_p95_trace = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Filter rows where trace_name contains the target callee patterns
targets = [
    "to_ts-auth-service",
    "to_ts-order-service",
    "to_ts-route-service",
    "to_ts-train-service",
    "to_ts-travel-service",
    "from_ts-auth-service",
    "from_ts-order-service",
    "from_ts-route-service",
    "from_ts-train-service",
    "from_ts-travel-service",
]
pattern = "(" + "|".join(map(re.escape, targets)) + ")"
mask_target = trace_window['trace_name'].str.contains(pattern, na=False)
trace_target = trace_window[mask_target].copy()

# 4) Attach global_p95 (computed from full series) to windowed target rows
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 5) Determine anomalies per row
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# 6) Aggregate per (cmdb_id, trace_name) for groups with >=1 anomaly
anoms = (
    trace_target[trace_target['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
    )
)

# Compute max value in window for these groups
max_in_window = (
    trace_target
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Merge and bring global_p95
summary = (
    anoms
    .merge(max_in_window, on=['cmdb_id', 'trace_name'], how='left')
    .merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')
)

# 7) Compute severity_ratio (None when global_p95 == 0 or NaN)
def compute_severity(r):
    gp = r['global_p95']
    if pd.isna(gp) or gp == 0:
        return None
    return r['max_value_in_window'] / gp

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# 8) Sort by anomaly_count desc, then severity_ratio desc (None treated as smallest for sorting)
summary['severity_sort'] = summary['severity_ratio'].fillna(-np.inf)
summary_top50 = (
    summary
    .sort_values(by=['anomaly_count', 'severity_sort'], ascending=[False, False])
    .drop(columns=['severity_sort'])
    .head(50)
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count',
             'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95', 'severity_ratio']]
)

# 9) Extract distinct caller cmdb_id -> callee service from trace_name for these groups
# Use regex to find 'to_ts-<service>' or 'from_ts-<service>' and extract the service name
def extract_callee(trace_name):
    m = re.search(r'(?:to_ts|from_ts)-([a-z0-9\-]+)', trace_name)
    return m.group(1) if m else None

caller_callee = summary_top50[['cmdb_id', 'trace_name']].copy()
caller_callee['callee_service'] = caller_callee['trace_name'].apply(extract_callee)
# Distinct mappings
caller_callee_distinct = caller_callee[['cmdb_id', 'callee_service']].dropna().drop_duplicates().reset_index(drop=True)

# Return compact outputs
summary_top50, caller_callee_distinct

```
Out[5]:
```
```python
summary = (
    "Trace-focused findings (timestamps UTC):\n"
    "- Incident window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39 UTC.\n"
    "- Filter: traces whose trace_name targets or originates from the candidate services (to_ts-*/from_ts-* for auth/order/route/train/travel) or where cmdb_id is one of those services.\n\n"
    "Top results (up to 50 groups returned):\n"
    "- Highest anomaly count: 4 occurrences for\n"
    "  * ts-admin-travel-service / trace.to_ts-travel-service.row_count (earliest anomaly 2024-01-23 06:54:00 UTC; max_in_window=11; global_p95=11; severity=1.00).\n"
    "- Strong severity spikes (max / global_p95 > 1):\n"
    "  * ts-auth-service: trace.from_ts-auth-service.duration_p95 and trace.to_ts-auth-service.duration_p95 — severity ≈ 1.45 (clear latency exceedance).\n"
    "  * root / trace.to_ts-travel-service.duration_p95 — severity ≈ 1.17.\n"
    "- Many groups show anomaly_count = 2 (multiple to_ts- and from_ts- traces for auth, order, travel and several 'root' traces), indicating repeated trace-level exceedances in these call paths.\n\n"
    "Caller → callee summary:\n"
    f"- Distinct caller→callee mappings observed among the reported groups: 22 unique pairs.\n"
    "- Examples: ts-admin-travel-service → travel-service, ts-auth-service → auth-service, root → travel-service, ts-order-service → order-service, etc.\n\n"
    "Interpretation / next steps:\n"
    "- The trace anomalies point to cross-service issues impacting auth and travel flows (both row_count and duration/p95 traces). The auth duration_p95 exceedance is the highest-severity latency signal to investigate first.\n"
    "- Also investigate the admin-travel → travel row_count anomaly (highest count) and related root traces to trace end-to-end request impacts across services."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                  trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio
17   ts-admin-travel-service        trace.to_ts-travel-service.row_count              4  2024-01-23 06:54:00+00:00            11.000000   11.000000        1.000000
19           ts-auth-service     trace.from_ts-auth-service.duration_p95              2  2024-01-23 07:01:00+00:00             0.006918    0.004762        1.452677
22           ts-auth-service       trace.to_ts-auth-service.duration_p95              2  2024-01-23 07:01:00+00:00             0.006918    0.004762        1.452677
13                      root     trace.to_ts-travel-service.duration_p95              2  2024-01-23 07:09:00+00:00             2.343556    2.007641        1.167318
1                       root       trace.to_ts-auth-service.duration_p95              2  2024-01-23 06:55:00+00:00             0.820397    0.725284        1.131139
..                       ...                                         ...            ...                        ...                  ...         ...             ...
54          ts-route-service  trace.from_ts-travel-service.duration_mean              1  2024-01-23 07:15:00+00:00             0.032565    0.029641        1.098658
93         ts-travel-service     trace.to_ts-route-service.duration_mean              1  2024-01-23 07:15:00+00:00             0.032565    0.029641        1.098658
88         ts-travel-service   trace.from_ts-travel-service.duration_p95              1  2024-01-23 07:06:00+00:00             0.209116    0.190387        1.098373
100        ts-travel-service     trace.to_ts-travel-service.duration_p95              1  2024-01-23 07:06:00+00:00             0.209116    0.190387        1.098373
62           ts-seat-service      trace.from_ts-travel-service.row_count              1  2024-01-23 07:05:00+00:00           123.000000  114.450000        1.074705

[50 rows x 7 columns],                     cmdb_id  callee_service
0   ts-admin-travel-service  travel-service
1           ts-auth-service    auth-service
2                      root  travel-service
3                      root    auth-service
4          ts-order-service   order-service
..                      ...             ...
17         ts-order-service  travel-service
18        ts-travel-service   order-service
19         ts-route-service  travel-service
20        ts-travel-service   route-service
21        ts-travel-service  travel-service

[22 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import re
import numpy as np
import pandas as pd

# Ensure timestamps are datetime UTC (reuse existing dataframes)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True)

# Candidate services (cmdb form) and canonical callee tokens
candidate_cmdbs = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]
candidate_tokens = [c.replace("ts-", "", 1) for c in candidate_cmdbs]  # e.g., 'auth-service'

# 1) Compute global P95 for full series (must be done BEFORE filtering)
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

global_p95_metric = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) From trace_window, extract rows that contain to_ts-<service> or from_ts-<service>
# Regex to extract callee token if present
def extract_callee_token(trace_name):
    m = re.search(r'(?:to_ts|from_ts)-([a-z0-9\-]+)', trace_name)
    return m.group(1) if m else None

trace_window['callee_token'] = trace_window['trace_name'].apply(extract_callee_token)

# Keep only rows where callee_token is one of candidate_tokens
trace_target = trace_window[trace_window['callee_token'].isin(candidate_tokens)].copy()

# Also add caller cmdb_id and callee_cmdb (full cmdb form)
trace_target['caller'] = trace_target['cmdb_id']
trace_target['callee_cmdb'] = 'ts-' + trace_target['callee_token']
trace_target['callee'] = trace_target['callee_token']  # canonical token like 'auth-service'

# 4) Attach global_p95 for traces (computed from full series)
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 5) Determine trace anomalies per row (rule: >= global_p95; if global_p95==0 then value>0)
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# 6) Aggregate per caller->callee pair (use caller cmdb_id and callee token)
group_cols = ['caller', 'callee']
# Anomaly aggregates
anoms = (
    trace_target[trace_target['is_anomaly']]
    .groupby(group_cols, as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# Max value in window per caller->callee (over trace_target rows)
max_in_window = (
    trace_target
    .groupby(group_cols + ['trace_name'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)
# We aggregated per caller+callee+trace_name; but requirement is per caller->callee (across trace_names).
# So take max across trace_names per caller->callee
max_across_traces = (
    max_in_window
    .groupby(['caller', 'callee'], as_index=False)
    .agg(max_value_in_window=('max_value_in_window', 'max'))
)

# For global_p95 to show a representative value per caller->callee, choose the max global_p95 among matched trace_names
gp_per_pair = (
    trace_target[['caller', 'callee', 'trace_name', 'global_p95']]
    .drop_duplicates()
    .groupby(['caller', 'callee'], as_index=False)
    .agg(global_p95=('global_p95', 'max'))
)

# Merge aggregates
summary = (
    anoms
    .merge(max_across_traces, on=['caller', 'callee'], how='left')
    .merge(gp_per_pair, on=['caller', 'callee'], how='left')
)

# 7) Compute severity_ratio = max_value_in_window / global_p95 (handle gp==0 or NaN)
def compute_sev(r):
    gp = r['global_p95']
    if pd.isna(gp) or gp == 0:
        return None
    return r['max_value_in_window'] / gp

summary['severity_ratio'] = summary.apply(compute_sev, axis=1)

# 8) Metric anomaly flag per callee service:
# For each candidate callee_cmdb, check if any metric row in metric_window for that cmdb_id is an anomaly
# Attach global_p95_metric to metric_window
metric_window = metric_window.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left')
metric_window['is_anomaly'] = np.where(
    metric_window['global_p95'] == 0,
    metric_window['value'] > 0,
    metric_window['value'] >= metric_window['global_p95']
)

# Compute a flag per callee token
metric_flags = []
for token in candidate_tokens:
    cmdb = 'ts-' + token
    dfc = metric_window[metric_window['cmdb_id'] == cmdb]
    flag = bool(dfc['is_anomaly'].any())
    metric_flags.append({'callee': token, 'callee_cmdb': cmdb, 'metric_anomaly_flag': flag})
metric_flags_df = pd.DataFrame(metric_flags)

# 9) Attach metric_anomaly_flag to summary (per callee)
summary = summary.merge(metric_flags_df[['callee', 'metric_anomaly_flag']], on='callee', how='left')

# 10) Sorting: by anomaly_count desc then severity_ratio desc. For sorting, treat None severity as -inf so real numbers come first.
summary['severity_sort'] = summary['severity_ratio'].fillna(-np.inf)
summary_top50 = (
    summary
    .sort_values(by=['anomaly_count', 'severity_sort'], ascending=[False, False])
    .drop(columns=['severity_sort'])
    .head(50)
    .loc[:, ['caller', 'callee', 'anomaly_count', 'earliest_anomaly_timestamp',
             'latest_anomaly_timestamp', 'max_value_in_window', 'global_p95',
             'severity_ratio', 'metric_anomaly_flag']]
)

# 11) Distinct list of callees (from candidate tokens) that have metric_anomaly_flag == True
callees_with_metric_anomaly = metric_flags_df[metric_flags_df['metric_anomaly_flag'] == True]['callee'].drop_duplicates().reset_index(drop=True)

# Compact outputs
summary_top50, callees_with_metric_anomaly

```
Out[6]:
```
```python
summary = (
    "Summary (UTC):\n"
    "- Window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39.\n"
    "- Method: global P95 computed from full series for traces and metrics; anomalies = value >= global P95 "
    "(or value>0 when global P95==0).\n\n"
    "Key trace caller→callee findings (top results):\n"
    "- ts-order-service → order-service: anomaly_count=12, max_trace=504 vs global_p95=487 (severity≈1.035).\n"
    "- ts-auth-service → auth-service: anomaly_count=12, max_trace=350 == global_p95 (severity=1.00).\n"
    "- root → travel-service: anomaly_count=6, max_trace=27 vs global_p95=24.9 (severity≈1.08).\n"
    "- ts-train-service → train-service and several other callers → travel-service show repeated anomalies (counts 6 and 3 in top rows).\n\n"
    "Metric correlation:\n"
    "- All five candidate callees had metric anomalies within the window: auth-service, order-service, route-service, train-service, travel-service (metric_anomaly_flag=True).\n\n"
    "Interpretation / recommended next steps:\n"
    "- There are concurrent trace and metric exceedances for multiple candidate services. Highest-priority investigation should target:\n"
    "  1) order-service traces (large row_count spikes and repeated anomalies)\n"
    "  2) auth-service traces (repeated anomalies) and related auth metrics\n"
    "  3) travel-related flows (root→travel and multiple callers→travel row_count anomalies) since both traces and metrics show issues.\n"
    "- Review trace spans for those caller→callee paths and corresponding service KPIs (CPU/disk/latency/memory) during the window to find root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   caller          callee  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio  metric_anomaly_flag
11       ts-order-service   order-service             12  2024-01-23 06:59:00+00:00 2024-01-23 07:07:00+00:00           504.000000  487.000000        1.034908                 True
6         ts-auth-service    auth-service             12  2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00           350.000000  350.000000        1.000000                 True
4                    root  travel-service              6  2024-01-23 07:02:00+00:00 2024-01-23 07:16:00+00:00            27.000000   24.900000        1.084337                 True
22       ts-train-service   train-service              6  2024-01-23 06:53:00+00:00 2024-01-23 06:57:00+00:00          2784.000000 2611.100000        1.066217                 True
18        ts-seat-service  travel-service              6  2024-01-23 07:05:00+00:00 2024-01-23 07:17:00+00:00           254.000000  240.700000        1.055256                 True
..                    ...             ...            ...                        ...                       ...                  ...         ...             ...                  ...
8        ts-basic-service   train-service              3  2024-01-23 06:56:00+00:00 2024-01-23 07:15:00+00:00           152.000000  145.350000        1.045752                 True
3                    root   train-service              3  2024-01-23 07:09:00+00:00 2024-01-23 07:13:00+00:00            19.000000   18.400000        1.032609                 True
21  ts-ticketinfo-service  travel-service              3  2024-01-23 07:05:00+00:00 2024-01-23 07:06:00+00:00           363.000000  351.600000        1.032423                 True
16       ts-route-service  travel-service              3  2024-01-23 07:02:00+00:00 2024-01-23 07:15:00+00:00           329.000000  319.500000        1.029734                 True
25      ts-travel-service   route-service              3  2024-01-23 07:02:00+00:00 2024-01-23 07:15:00+00:00           329.000000  319.500000        1.029734                 True

[30 rows x 9 columns], 0      auth-service
1     order-service
2     route-service
3     train-service
4    travel-service
Name: callee, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import re
import numpy as np
import pandas as pd

# Ensure trace_df exists and timestamps are parsed as UTC from Unix seconds
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Compute global P95 per (cmdb_id, trace_name) using the full series (before filtering)
try:
    global_p95_trace
except NameError:
    global_p95_trace = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

# Filter to window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# Extract callee token from trace_name when pattern to_ts-<service> or from_ts-<service> exists
trace_window['callee_token'] = trace_window['trace_name'].str.extract(r'(?:to_ts|from_ts)-([a-z0-9\-]+)', expand=False)

# Keep only rows where callee_token is one of the target services
target_tokens = {'auth-service', 'order-service', 'route-service', 'train-service', 'travel-service'}
trace_target = trace_window[trace_window['callee_token'].isin(target_tokens)].copy()

# Attach global_p95 computed from full series
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# Determine anomaly per row:
# - if global_p95 == 0 => anomaly if value > 0
# - else anomaly if value >= global_p95
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# Compute severity_ratio:
# - if global_p95 > 0: severity = value / global_p95 (use tiny eps to avoid exact-zero edge)
# - if global_p95 == 0 and value > 0: set a very large number (1e9)
# - else None
def compute_severity(row):
    gp = row['global_p95']
    val = row['value']
    if pd.isna(gp):
        return None
    if gp > 0:
        return float(max(val, gp * 1e-12) / gp)
    # gp == 0
    if val > 0:
        return 1e9
    return None

trace_target['severity_ratio'] = trace_target.apply(compute_severity, axis=1)

# Keep only anomalous rows, sort by timestamp ascending, select required columns, limit to first 200
anomalous_rows = trace_target[trace_target['is_anomaly']].copy()
anomalous_rows_sorted = anomalous_rows.sort_values('timestamp').loc[:, [
    'timestamp',         # UTC datetime
    'cmdb_id',           # caller
    'trace_name',
    'value',
    'global_p95',
    'severity_ratio'
]]

anomalous_rows_head = anomalous_rows_sorted.head(200)

anomalous_rows_head

```
Out[7]:
```
Summary (plain English, timestamps UTC):

- Incident window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39 UTC.
- Filtered trace rows that target or originate from the five candidate services and marked anomalies (value >= global P95, or value>0 when global P95 == 0).
- Results returned: 129 anomalous trace rows (sorted by timestamp ascending; limit 200 applied).

Key observations and examples (chronological highlights):
- Earliest anomalies at ~06:53 involve train-service traces (duration_p95/duration_mean) with severity ≈1.03–1.05.
- At 06:54 root → auth-service row_count shows a small exceedance (value 88 vs global_p95 87.8; severity ≈1.002).
- Multiple auth-related duration_p95 and duration_mean traces exceed P95 repeatedly (e.g., ts-auth-service trace.from_ts-auth-service.duration_p95 at 07:17 with severity ≈1.45).
- Several travel-related spikes occur later in the window (notably around 07:06–07:17):
  - ts-admin-travel-service → travel-service duration_p95 (07:17) severity ≈1.48 and duration_mean severity ≈1.80 (large relative spike).
  - Other callers (ts-preserve-service, ts-travel2-service, ts-seat-service, ts-route-service, etc.) show travel-related duration/row_count exceedances with severity ratios >1.0.
- Many anomalies are modest exceedances (severity near 1.03–1.17), but a few traces show stronger deviations (>1.4 and up to ~1.8).

Interpretation / suggested next steps:
- The anomalous traces implicate cross-service call paths involving auth and travel flows, with repeated auth duration_p95 exceedances and stronger travel-related spikes near 07:17.
- Prioritize investigating:
  1) auth-service latency traces (duration_p95) for root cause of sustained exceedances,
  2) travel-related duration_mean/p95 spikes (admin-travel → travel and other callers) because of high severity ratios,
  3) callers with large row_count spikes (e.g., order-service and root→travel) to understand traffic/throughput changes.
- Correlate these trace anomalies with corresponding service metrics (CPU, disk I/O, memory, socket, etc.) and logs in the same time window to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                  cmdb_id                                 trace_name     value  global_p95  severity_ratio
85   2024-01-23 06:53:00+00:00         ts-train-service   trace.from_ts-train-service.duration_p95  0.001842    0.001776        1.036872
89   2024-01-23 06:53:00+00:00         ts-train-service  trace.from_ts-travel-service.duration_p95  0.007426    0.007085        1.048120
93   2024-01-23 06:53:00+00:00         ts-train-service     trace.to_ts-train-service.duration_p95  0.001842    0.001776        1.036872
109  2024-01-23 06:53:00+00:00        ts-travel-service     trace.to_ts-train-service.duration_p95  0.007426    0.007085        1.048120
127  2024-01-23 06:54:00+00:00                     root         trace.to_ts-auth-service.row_count 88.000000   87.800000        1.002278
...                        ...                      ...                                        ...       ...         ...             ...
2653 2024-01-23 07:17:00+00:00          ts-auth-service    trace.from_ts-auth-service.duration_p95  0.006918    0.004762        1.452677
2649 2024-01-23 07:17:00+00:00  ts-admin-travel-service    trace.to_ts-travel-service.duration_p95  1.086738    0.733627        1.481323
2648 2024-01-23 07:17:00+00:00  ts-admin-travel-service   trace.to_ts-travel-service.duration_mean  0.425062    0.236126        1.800153
2688 2024-01-23 07:17:00+00:00      ts-preserve-service   trace.to_ts-travel-service.duration_mean  0.720679    0.579913        1.242736
2765 2024-01-23 07:17:00+00:00       ts-travel2-service     trace.to_ts-train-service.duration_p95  0.007315    0.005951        1.229176

[129 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import re
import numpy as np
import pandas as pd

# Ensure timestamps parsed as UTC from Unix seconds
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidate_cmdbs = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]
candidate_tokens = [c.replace("ts-", "", 1) for c in candidate_cmdbs]  # e.g., 'auth-service'

# 1) Global P95 thresholds computed from full series (before window filtering)
try:
    global_p95_trace
except NameError:
    global_p95_trace = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

try:
    global_p95_metric
except NameError:
    global_p95_metric = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) From trace_window, keep only rows where trace_name contains 'to_ts-<service>' and
#    both caller (cmdb_id) and callee are among the five candidates.
trace_window['callee_token'] = trace_window['trace_name'].str.extract(r'to_ts-([a-z0-9\-]+)', expand=False)
# Keep rows where caller is candidate and callee token in candidate_tokens
trace_target = trace_window[
    (trace_window['cmdb_id'].isin(candidate_cmdbs)) &
    (trace_window['callee_token'].isin(candidate_tokens))
].copy()

# Add caller and callee cmdb forms
trace_target['caller'] = trace_target['cmdb_id']
trace_target['callee'] = 'ts-' + trace_target['callee_token']

# 4) Attach global_p95 for each trace row (computed from full series)
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 5) Determine row-level anomaly using the rule (global thresholds computed earlier)
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# 6) Aggregate per caller->callee pair:
# - anomaly_count, earliest_anomaly_timestamp, latest_anomaly_timestamp (from anomalous rows)
anom_rows = trace_target[trace_target['is_anomaly']].copy()
anoms = (
    anom_rows
    .groupby(['caller', 'callee'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# - max_trace_value_in_window per pair (consider all trace_target rows)
max_vals = (
    trace_target
    .groupby(['caller', 'callee'], as_index=False)
    .agg(max_trace_value_in_window=('value', 'max'))
)

# To choose a representative trace_global_p95 for the pair, pick the global_p95 of the trace_name that produced the max value.
# Find trace_name index of max per pair
idx = trace_target.groupby(['caller', 'callee'])['value'].idxmax().dropna().astype(int)
rep_rows = trace_target.loc[idx, ['caller', 'callee', 'trace_name', 'global_p95']].copy()
rep_rows = rep_rows.rename(columns={'global_p95': 'trace_global_p95', 'trace_name': 'trace_name_of_max'})

# Merge aggregates
summary = anoms.merge(max_vals, on=['caller', 'callee'], how='left').merge(rep_rows[['caller','callee','trace_global_p95']], on=['caller','callee'], how='left')

# 7) Compute severity_ratio = max_trace_value_in_window / trace_global_p95; handle zero or NaN gracefully (use None)
def sev_ratio(row):
    gp = row['trace_global_p95']
    mv = row['max_trace_value_in_window']
    if pd.isna(gp) or gp == 0:
        return None
    return float(mv / gp)

summary['severity_ratio'] = summary.apply(sev_ratio, axis=1)

# 8) Compute callee_metric_anomaly_flag for each callee: True if metric_window has any KPI anomaly for that callee
# Attach global_p95_metric to metric_window and evaluate anomalies per metric row
metric_window = metric_window.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left')
metric_window['is_anomaly'] = np.where(
    metric_window['global_p95'] == 0,
    metric_window['value'] > 0,
    metric_window['value'] >= metric_window['global_p95']
)

metric_flags = (
    metric_window
    .groupby('cmdb_id', as_index=False)
    .agg(callee_metric_anomaly_flag=('is_anomaly', 'any'))
)
# Normalize to cmdb ids; keep only candidate callees
metric_flags = metric_flags[metric_flags['cmdb_id'].isin(candidate_cmdbs)].rename(columns={'cmdb_id':'callee'})

# Merge metric flag into summary
summary = summary.merge(metric_flags[['callee','callee_metric_anomaly_flag']], on='callee', how='left')
summary['callee_metric_anomaly_flag'] = summary['callee_metric_anomaly_flag'].fillna(False)

# 9) Sort and limit results (top 20 by anomaly_count desc then severity_ratio desc)
summary_sorted = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])
summary_top20 = summary_sorted.loc[:, [
    'caller', 'callee', 'anomaly_count', 'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
    'max_trace_value_in_window', 'trace_global_p95', 'severity_ratio', 'callee_metric_anomaly_flag'
]].head(20)

# 10) Distinct list of callee services (among the five) that have callee_metric_anomaly_flag == True
callees_with_metric_anomaly = metric_flags[metric_flags['callee_metric_anomaly_flag']].copy()
callees_with_metric_anomaly = callees_with_metric_anomaly['callee'].drop_duplicates().reset_index(drop=True)

# Return compact outputs
summary_top20, callees_with_metric_anomaly

```
Out[8]:
```
```python
summary = (
    "Incident window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39 UTC.\n"
    "Filter: traces of form 'to_ts-<service>' where both caller and callee are among the five candidates (auth, order, route, train, travel).\n\n"
    "Key findings:\n"
    "- Highest trace anomaly counts: ts-order-service→ts-order-service (anomaly_count=6, earliest=2024-01-23 06:59:00 UTC, latest=07:07:00 UTC, max=504, global_p95=487, severity≈1.035) and ts-auth-service→ts-auth-service (anomaly_count=6, earliest=06:54:00, latest=07:17:00, max=350, global_p95=350, severity=1.00).\n"
    "- Several other caller→callee pairs show repeated anomalies (anomaly_count=3) including calls involving train, route and travel services with moderate severity ratios (~1.03–1.07).\n"
    "- All five candidate callee services (auth-service, order-service, route-service, train-service, travel-service) had metric anomalies during the window (callee_metric_anomaly_flag=True).\n\n"
    "Interpretation and next steps:\n"
    "- Correlated trace and metric anomalies point to concurrent issues across multiple services; prioritize investigation of order-service and auth-service traces and their service KPIs first, then investigate train/route/travel flows.\n"
    "- Check detailed spans and corresponding KPIs (CPU, disk I/O, memory, latency) and logs around the listed timestamps to identify root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              caller             callee  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_trace_value_in_window  trace_global_p95  severity_ratio  callee_metric_anomaly_flag
1   ts-order-service   ts-order-service              6  2024-01-23 06:59:00+00:00 2024-01-23 07:07:00+00:00                 504.000000        487.000000        1.034908                        True
0    ts-auth-service    ts-auth-service              6  2024-01-23 06:54:00+00:00 2024-01-23 07:17:00+00:00                 350.000000        350.000000        1.000000                        True
4  ts-travel-service   ts-order-service              3  2024-01-23 07:05:00+00:00 2024-01-23 07:07:00+00:00                  61.000000         57.200000        1.066434                        True
3   ts-train-service   ts-train-service              3  2024-01-23 06:53:00+00:00 2024-01-23 06:57:00+00:00                2784.000000       2611.100000        1.066217                        True
6  ts-travel-service   ts-train-service              3  2024-01-23 06:53:00+00:00 2024-01-23 07:05:00+00:00                 232.000000        221.550000        1.047168                        True
2   ts-route-service   ts-route-service              3  2024-01-23 06:57:00+00:00 2024-01-23 07:06:00+00:00                2218.000000       2130.600000        1.041021                        True
5  ts-travel-service   ts-route-service              3  2024-01-23 07:02:00+00:00 2024-01-23 07:15:00+00:00                 329.000000        319.500000        1.029734                        True
7  ts-travel-service  ts-travel-service              3  2024-01-23 07:05:00+00:00 2024-01-23 07:17:00+00:00                1731.000000       1695.850000        1.020727                        True, 0      ts-auth-service
1     ts-order-service
2     ts-route-service
3     ts-train-service
4    ts-travel-service
Name: callee, dtype: object)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import re

# Ensure timestamps parsed as UTC from Unix seconds (reuse existing variables if present)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]
candidate_tokens = [c.replace("ts-", "", 1) for c in candidates]  # e.g., 'auth-service'

# 1) Global P95 thresholds computed from full series (before window filtering)
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

global_p95_metric = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# 3) Keep only trace rows with 'to_ts-<service>' where <service> is one of the five candidates
trace_window['callee_token'] = trace_window['trace_name'].str.extract(r'to_ts-([a-z0-9\-]+)', expand=False)
trace_target = trace_window[trace_window['callee_token'].isin(candidate_tokens)].copy()
# derive callee cmdb form and caller
trace_target['caller'] = trace_target['cmdb_id']
trace_target['callee'] = 'ts-' + trace_target['callee_token']

# 4) Attach global_p95 for traces (computed from full series)
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 5) Mark anomalous trace rows: value >= global_p95 OR (global_p95==0 and value>0)
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# 6) Compute per-callee aggregations
# compute per-row severity (handle zero gp by using a very large number when value>0)
def row_severity(val, gp):
    if pd.isna(gp):
        return None
    if gp > 0:
        return float(val / gp)
    # gp == 0
    if val > 0:
        return 1e9
    return None

trace_target['row_severity'] = trace_target.apply(lambda r: row_severity(r['value'], r['global_p95']), axis=1)

# total_callee_anomaly_count
anom_rows = trace_target[trace_target['is_anomaly']].copy()

# Aggregate per callee
agg_count = (
    anom_rows
    .groupby('callee', as_index=False)
    .agg(total_callee_anomaly_count=('is_anomaly', 'sum'),
         earliest_callee_anomaly_timestamp=('timestamp', 'min'))
)

# earliest may be missing for some candidates; ensure all candidates present
agg_count = pd.DataFrame({'callee': candidates}).merge(agg_count, on='callee', how='left')

# max_trace_severity_as_callee (max of row_severity among anomalous rows for that callee)
max_trace_sev = (
    anom_rows
    .groupby('callee', as_index=False)
    .agg(max_trace_severity_as_callee=('row_severity', 'max'))
)

# total_distinct_callers_reporting_anomalies
distinct_callers = (
    anom_rows[['callee', 'caller']].drop_duplicates()
    .groupby('callee', as_index=False)
    .agg(total_distinct_callers_reporting_anomalies=('caller', 'nunique'))
)

# Merge all trace aggregates
trace_agg = agg_count.merge(max_trace_sev, on='callee', how='left').merge(distinct_callers, on='callee', how='left')

# Fill NaNs for counts/nums with 0 and keep earliest timestamp as NaT if none
trace_agg['total_callee_anomaly_count'] = trace_agg['total_callee_anomaly_count'].fillna(0).astype(int)
trace_agg['total_distinct_callers_reporting_anomalies'] = trace_agg['total_distinct_callers_reporting_anomalies'].fillna(0).astype(int)

# 7) Metric-side: for each candidate callee compute max_metric_severity
# Attach global_p95_metric to metric_window
metric_window = metric_window.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('','_kpi'))
# compute max value per KPI within the window (max_value_in_window)
max_per_kpi = (
    metric_window
    .groupby(['cmdb_id', 'kpi_name', 'global_p95'], as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# compute KPI severity
def kpi_sev(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp):
        return None
    if gp > 0:
        return float(mv / gp)
    if mv > 0:
        return 1e9
    return None

max_per_kpi['kpi_severity'] = max_per_kpi.apply(kpi_sev, axis=1)

# For each callee (cmdb), get max over KPIs
max_metric_sev = (
    max_per_kpi
    .groupby('cmdb_id', as_index=False)
    .agg(max_metric_severity=('kpi_severity', 'max'))
)
# Ensure all candidates present
max_metric_sev = pd.DataFrame({'cmdb_id': candidates}).merge(max_metric_sev, left_on='cmdb_id', right_on='cmdb_id', how='left')

# 8) Merge metric severity into trace_agg (align by callee cmdb)
trace_agg = trace_agg.rename(columns={'callee': 'callee_service'})
trace_agg = trace_agg.merge(max_metric_sev.rename(columns={'cmdb_id':'callee_service'}), on='callee_service', how='left')

# 9) Final formatting: select requested columns and sort by earliest timestamp ascending
result = trace_agg.loc[:, [
    'callee_service',
    'total_callee_anomaly_count',
    'earliest_callee_anomaly_timestamp',
    'max_trace_severity_as_callee',
    'total_distinct_callers_reporting_anomalies',
    'max_metric_severity'
]]

# Sort by earliest timestamp ascending (NaT at end)
result_sorted = result.sort_values(by=['earliest_callee_anomaly_timestamp']).reset_index(drop=True).head(5)

# Also produce list of callee services that have any metric severity > 1 or severity is large (not-NaN)
callees_with_metric_issue = result_sorted[result_sorted['max_metric_severity'].notna() & (result_sorted['max_metric_severity'] > 1)]['callee_service'].reset_index(drop=True)

# Return compact outputs
result_sorted, callees_with_metric_issue

```
Out[9]:
```
Summary (timestamps UTC):

- Window: 2024-01-23 06:50:39 — 2024-01-23 07:20:39.

Per-callee summary (one row per candidate service):

- ts-train-service
  - total_callee_anomaly_count: 15
  - earliest anomaly: 2024-01-23 06:53:00 UTC
  - max trace severity (as callee): 1.229
  - distinct callers reporting anomalies: 5
  - max metric severity (across KPIs): 1.033

- ts-auth-service
  - total_callee_anomaly_count: 12
  - earliest anomaly: 2024-01-23 06:54:00 UTC
  - max trace severity (as callee): 1.453
  - distinct callers reporting anomalies: 2
  - max metric severity (across KPIs): 1.054

- ts-order-service
  - total_callee_anomaly_count: 24
  - earliest anomaly: 2024-01-23 06:54:00 UTC
  - max trace severity (as callee): 1.370
  - distinct callers reporting anomalies: 7
  - max metric severity (across KPIs): 2.548

- ts-travel-service
  - total_callee_anomaly_count: 24
  - earliest anomaly: 2024-01-23 06:54:00 UTC
  - max trace severity (as callee): 1.800
  - distinct callers reporting anomalies: 6
  - max metric severity (across KPIs): 1.144

- ts-route-service
  - total_callee_anomaly_count: 15
  - earliest anomaly: 2024-01-23 06:57:00 UTC
  - max trace severity (as callee): 1.280
  - distinct callers reporting anomalies: 5
  - max metric severity (across KPIs): 3.774

Which callees show metric problems?
- All five candidate services show metric-side severity > 1 in the incident window: auth, order, route, train, travel. (Order and Route show the largest metric severities.)

Interpretation / recommended priorities:
- Order-service and Travel-service have the largest total trace anomaly counts (24 each); Order and Route show the largest metric severity ratios (Order ≈ 2.55, Route ≈ 3.77).
- Travel has the largest trace severity (≈1.80) indicating strong trace-level spikes.
- Recommended next steps: investigate order-service and route-service KPIs first (highest metric deviations), and investigate travel- and auth-related traces for latency spikes (highest trace severities). Correlate these trace spikes with the specific KPIs (disk I/O, latency, memory, socket) and logs around the earliest timestamps (06:53–06:57 UTC) to find the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(      callee_service  total_callee_anomaly_count earliest_callee_anomaly_timestamp  max_trace_severity_as_callee  total_distinct_callers_reporting_anomalies  max_metric_severity
0   ts-train-service                          15         2024-01-23 06:53:00+00:00                      1.229176                                           5             1.032745
1    ts-auth-service                          12         2024-01-23 06:54:00+00:00                      1.452677                                           2             1.054152
2   ts-order-service                          24         2024-01-23 06:54:00+00:00                      1.369962                                           7             2.547879
3  ts-travel-service                          24         2024-01-23 06:54:00+00:00                      1.800153                                           6             1.144393
4   ts-route-service                          15         2024-01-23 06:57:00+00:00                      1.279980                                           5             3.774242, 0     ts-train-service
1      ts-auth-service
2     ts-order-service
3    ts-travel-service
4     ts-route-service
Name: callee_service, dtype: object)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import re

# Re-load trace.csv to ensure timestamp parsing from Unix seconds to UTC datetimes (rule 10)
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidate_cmdbs = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]
candidate_tokens = [c.replace("ts-", "", 1) for c in candidate_cmdbs]

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series BEFORE filtering (rule 9)
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(global_p95=('value', lambda s: s.quantile(0.95)))
)

# 2) Filter trace rows to the incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Select rows where trace_name contains 'to_ts-<service>' and both caller and callee are among the five candidates
trace_window['callee_token'] = trace_window['trace_name'].str.extract(r'to_ts-([a-z0-9\-]+)', expand=False)
trace_window['callee_cmdb'] = trace_window['callee_token'].apply(lambda t: 'ts-' + t if pd.notna(t) else None)

mask = trace_window['callee_token'].notna() & \
       trace_window['cmdb_id'].isin(candidate_cmdbs) & \
       trace_window['callee_cmdb'].isin(candidate_cmdbs)

trace_target = trace_window[mask].copy()

# 4) Attach global_p95 values computed from the full series
trace_target = trace_target.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 5) Mark anomalies per row (value >= global_p95, or value>0 when global_p95==0)
trace_target['is_anomaly'] = np.where(
    trace_target['global_p95'] == 0,
    trace_target['value'] > 0,
    trace_target['value'] >= trace_target['global_p95']
)

# 6) Compute severity_ratio per row (handle zero P95 by using a large number when value>0)
def compute_severity(val, gp):
    if pd.isna(gp):
        return None
    if gp > 0:
        return float(val / gp)
    # gp == 0
    if val > 0:
        return 1e9
    return None

trace_target['severity_ratio'] = trace_target.apply(lambda r: compute_severity(r['value'], r['global_p95']), axis=1)

# 7) Keep only anomalous rows, sort by timestamp ascending, select required columns, limit to first 200
anomalous_rows = trace_target[trace_target['is_anomaly']].copy()
anomalous_rows_sorted = anomalous_rows.sort_values('timestamp').loc[:, [
    'timestamp',
    'cmdb_id',            # caller
    'callee_token',       # callee (extracted <service>)
    'trace_name',
    'value',
    'global_p95',
    'severity_ratio'
]]

anomalous_rows_head = anomalous_rows_sorted.head(200)

# Display the compact result variable (rule 2)
anomalous_rows_head

```
Out[10]:
```
```python
summary = (
    "Chronological anomalous caller→callee traces (UTC) — concise findings:\n"
    "- Window: 2024-01-23 06:50:39 → 2024-01-23 07:20:39 UTC. Returned anomalous 'to_ts-<service>' rows (caller and callee both in the five candidates).\n"
    "- Number of anomalous rows returned (chronological sample): 30 (limited to first 200 rows).\n"
    "- Earliest anomalies: around 06:53 UTC involving train-service traces (to_ts-train-service duration_p95/mean).\n"
    "- Early important event: 06:54 UTC ts-auth-service → auth-service row_count = 350 (equals global P95). Indicates auth traffic spike.\n"
    "- Repeated patterns: many callers (including travel-service and root) invoked train, order, route, travel; repeated anomalies appear from ~06:53 through ~07:17.\n"
    "- Highest notable severities in this set:\n"
    "  * ts-auth-service → auth-service: duration_p95 at 07:17 UTC, severity ≈ 1.45 (clear latency exceedance).\n"
    "  * Several travel-related calls show elevated durations near 07:06–07:17 UTC.\n"
    "- Interpretation: trace events show widespread, time-ordered exceedances across multiple call paths; auth-service shows a clear latency spike (high severity) and travel-related calls show multiple elevated durations and counts.\n"
    "- Suggested next steps: focus inspection on auth-service latency spans (around 07:17) and travel-related traces around 07:06–07:17; correlate these chronological trace events with the callee service metrics (CPU, disk I/O, memory, socket) and service logs at the same timestamps to identify the downstream faulty service or root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp            cmdb_id    callee_token                                trace_name      value  global_p95  severity_ratio
13  2024-01-23 06:53:00+00:00   ts-train-service   train-service    trace.to_ts-train-service.duration_p95   0.001842    0.001776        1.036872
25  2024-01-23 06:53:00+00:00  ts-travel-service   train-service    trace.to_ts-train-service.duration_p95   0.007426    0.007085        1.048120
35  2024-01-23 06:54:00+00:00    ts-auth-service    auth-service        trace.to_ts-auth-service.row_count 350.000000  350.000000        1.000000
88  2024-01-23 06:55:00+00:00  ts-travel-service   train-service   trace.to_ts-train-service.duration_mean   0.004711    0.004703        1.001691
108 2024-01-23 06:56:00+00:00   ts-train-service   train-service   trace.to_ts-train-service.duration_mean   0.001254    0.001245        1.007214
..                        ...                ...             ...                                       ...        ...         ...             ...
465 2024-01-23 07:07:00+00:00  ts-travel-service   order-service    trace.to_ts-order-service.duration_p95   0.041163    0.036765        1.119643
572 2024-01-23 07:15:00+00:00  ts-travel-service   route-service   trace.to_ts-route-service.duration_mean   0.032565    0.029641        1.098658
573 2024-01-23 07:15:00+00:00  ts-travel-service   route-service    trace.to_ts-route-service.duration_p95   0.211172    0.210586        1.002785
617 2024-01-23 07:17:00+00:00    ts-auth-service    auth-service     trace.to_ts-auth-service.duration_p95   0.006918    0.004762        1.452677
644 2024-01-23 07:17:00+00:00  ts-travel-service  travel-service  trace.to_ts-travel-service.duration_mean   0.027832    0.025909        1.074214

[30 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np

# Load CSVs (reloading to ensure fresh state)
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")

# Parse timestamps as UTC datetimes from Unix seconds
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

# Target service
service = 'ts-travel-service'

# 1) Compute global P95 per (cmdb_id, kpi_name) from the full metric series (BEFORE filtering)
global_p95_metric = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric rows to the incident window and the target service
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end) & (metric_df['cmdb_id'] == service)].copy()

# Attach global_p95 to windowed metric rows
metric_window = metric_window.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalies per row:
metric_window['is_anomaly'] = np.where(
    metric_window['global_p95'] == 0,
    metric_window['value'] > 0,
    metric_window['value'] >= metric_window['global_p95']
)

# Aggregate per kpi_name
# - anomaly_count (count of anomalous rows)
# - earliest_anomaly_timestamp, latest_anomaly_timestamp (from anomalous rows)
anom_rows_metric = metric_window[metric_window['is_anomaly']].copy()

agg_anom = (
    anom_rows_metric
    .groupby('kpi_name', as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# Max value in window per KPI (even if no anomalies)
max_in_window = (
    metric_window
    .groupby('kpi_name', as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Representative global_p95 per KPI (from precomputed global_p95_metric for this service)
gp_for_service = global_p95_metric[global_p95_metric['cmdb_id'] == service].rename(columns={'global_p95':'global_p95'})

# Merge all summaries
metric_kpi_summary = (
    pd.DataFrame({'kpi_name': metric_window['kpi_name'].unique()})  # ensure all KPIs in window are included
    .merge(agg_anom, on='kpi_name', how='left')
    .merge(max_in_window, on='kpi_name', how='left')
    .merge(gp_for_service[['kpi_name','global_p95']], on='kpi_name', how='left')
)

# Fill missing anomaly counts with 0
metric_kpi_summary['anomaly_count'] = metric_kpi_summary['anomaly_count'].fillna(0).astype(int)

# Compute severity_ratio = max_value_in_window / global_p95 (handle gp==0 or NaN -> None)
def compute_sev(mv, gp):
    if pd.isna(gp) or gp == 0:
        return None
    return float(mv / gp)

metric_kpi_summary['severity_ratio'] = metric_kpi_summary.apply(
    lambda r: compute_sev(r['max_value_in_window'], r['global_p95']), axis=1
)

# Sort by severity_ratio descending and limit to top 20
metric_kpi_top20 = metric_kpi_summary.sort_values(by='severity_ratio', ascending=False).head(20).loc[
    :, ['kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
         'max_value_in_window', 'global_p95', 'severity_ratio']
].reset_index(drop=True)

# 3) Process logs
# Compute global P95 per (cmdb_id, log_name) from full log series BEFORE filtering
global_p95_log = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Filter log rows to the incident window and target service
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end) & (log_df['cmdb_id'] == service)].copy()

# Aggregate per log_name in the window: count, min_timestamp, max_timestamp, max_value_in_window
log_agg_window = (
    log_window
    .groupby('log_name', as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        max_value_in_window=('value', 'max')
    )
)

# Attach global_p95 (from full log series) per log_name for this service
gp_log_service = global_p95_log[global_p95_log['cmdb_id'] == service][['log_name','global_p95']]
log_summary = log_agg_window.merge(gp_log_service, on='log_name', how='left')

# 4) Identify log rows indicating errors: log_name contains 'error' (case-insensitive) and value>0
error_mask = log_window['log_name'].str.contains('error', case=False, na=False) & (log_window['value'] > 0)
error_log_rows = log_window[error_mask].loc[:, ['timestamp', 'log_name', 'value']].sort_values('timestamp').head(20).reset_index(drop=True)

# Final compact outputs
metric_kpi_top20, log_summary, error_log_rows

```
Out[11]:
```
Summary (plain English, timestamps UTC):

- Window: 2024-01-23 06:50:39 → 2024-01-23 07:20:39.

Metric findings for ts-travel-service (global P95 computed from full series, anomalies defined as value >= P95 or value>0 when P95==0):
- workload: anomaly_count=2, earliest=2024-01-23 06:53, latest=07:02, max=11.133, global_p95≈9.728, severity≈1.144 (largest metric severity).
- cpu: anomaly_count=2, earliest=07:01, latest=07:02, max≈3.430, global_p95≈3.203, severity≈1.071.
- socket: anomaly_count=2, earliest=07:06, latest=07:15, max≈18.983, global_p95≈18.08, severity≈1.050.
- diskio: anomaly_count=2, earliest=07:02, latest=07:05, max≈60,972.97, global_p95≈60,007.77, severity≈1.016.
- latency-50: anomaly_count=2, earliest=07:07, latest=07:13, max≈0.01996, global_p95≈0.01974, severity≈1.011.
- latency-90: anomaly_count=2, earliest=07:09, latest=07:17, max≈0.45562, global_p95≈0.45359, severity≈1.004.
- mem: anomaly_count=2, earliest=07:15, latest=07:17, max≈272,246,033.07, global_p95≈272,201,550.51, severity≈1.00016.
- error (metric): anomaly_count=0 (global_p95==0 and max value 0).

Log findings for ts-travel-service in the same window:
- log.row_count: count=20, time range 2024-01-23 06:53 → 07:17, max_value_in_window=1929, global_p95≈1910.95 (slight exceedance).
- log.error_count: count=20, time range 06:53 → 07:17, max_value_in_window=0, global_p95=0.
- No log rows with error-indicating names and value>0 were found in the window.

Interpretation and recommended next steps:
- The strongest metric signals are workload and CPU (highest severity ratios), with disk I/O and socket elevated as well. Workload spike appears earliest (~06:53); CPU and diskio spikes around 07:01–07:05; socket increases extend through ~07:06–07:15. Latency spikes follow (07:07–07:17).
- Logs do not show error counts >0, but row_count slightly exceeded its P95, indicating increased traffic/throughput.
- Actionable next steps: investigate ts-travel-service KPIs around the listed timestamps (start with workload and CPU, then disk I/O and socket), correlate with traces and upstream callers during 06:53–07:17 to find the upstream trigger or the downstream resource bottleneck.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window       global_p95  severity_ratio
0    workload              2  2024-01-23 06:53:00+00:00 2024-01-23 07:02:00+00:00            11.133143         9.728423        1.144393
1         cpu              2  2024-01-23 07:01:00+00:00 2024-01-23 07:02:00+00:00             3.429573         3.203012        1.070734
2      socket              2  2024-01-23 07:06:00+00:00 2024-01-23 07:15:00+00:00            18.983333        18.080000        1.049963
3      diskio              2  2024-01-23 07:02:00+00:00 2024-01-23 07:05:00+00:00         60972.969086     60007.766282        1.016085
4  latency-50              2  2024-01-23 07:07:00+00:00 2024-01-23 07:13:00+00:00             0.019957         0.019741        1.010949
5  latency-90              2  2024-01-23 07:09:00+00:00 2024-01-23 07:17:00+00:00             0.455620         0.453590        1.004475
6         mem              2  2024-01-23 07:15:00+00:00 2024-01-23 07:17:00+00:00     272246033.066667 272201550.506667        1.000163
7       error              0                        NaT                       NaT             0.000000         0.000000             NaN,           log_name  count             min_timestamp             max_timestamp  max_value_in_window  global_p95
0  log.error_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00                    0    0.000000
1    log.row_count     20 2024-01-23 06:53:00+00:00 2024-01-23 07:17:00+00:00                 1929 1910.950000, Empty DataFrame
Columns: [timestamp, log_name, value]
Index: [])```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np

# Load metric.csv and parse timestamps as UTC datetimes (Unix seconds)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure global P95 per (cmdb_id, kpi_name) is computed from the full series BEFORE filtering
global_p95_metric = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Incident window (inclusive)
start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
end = pd.to_datetime("2024-01-23 07:20:39", utc=True)

# Target service and KPI list
service = 'ts-travel-service'
kpis = ['workload', 'cpu', 'socket', 'diskio', 'latency-90', 'mem']

# Filter metric rows to the incident window and the target service and selected KPIs
metric_window = metric_df[
    (metric_df['cmdb_id'] == service) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    (metric_df['kpi_name'].isin(kpis))
].copy()

# Attach KPI global_p95 (computed from full series)
metric_window = metric_window.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalous points per rule
metric_window['is_anomaly'] = np.where(
    metric_window['global_p95'] == 0,
    metric_window['value'] > 0,
    metric_window['value'] >= metric_window['global_p95']
)

# Helper to compute segments of consecutive anomaly points (consecutive minutes)
segments_list = []
for kpi, grp in metric_window.groupby('kpi_name'):
    # sort by timestamp
    grp = grp.sort_values('timestamp').reset_index(drop=True)
    # keep only anomalous rows
    anom = grp[grp['is_anomaly']].copy()
    if anom.empty:
        continue
    # compute time difference to previous row
    anom['ts_diff'] = anom['timestamp'].diff()
    # start new segment when diff is not 1 minute (or first row)
    one_min = pd.Timedelta(minutes=1)
    anom['new_segment'] = (anom['ts_diff'].isna()) | (anom['ts_diff'] != one_min)
    anom['segment_id'] = anom['new_segment'].cumsum()
    # aggregate per segment
    seg_agg = (
        anom
        .groupby('segment_id', as_index=False)
        .agg(
            kpi_name=('kpi_name', 'first'),
            segment_start_timestamp=('timestamp', 'min'),
            segment_end_timestamp=('timestamp', 'max'),
            segment_length_in_minutes=('timestamp', 'count'),
            max_value_in_segment=('value', 'max'),
            KPI_global_P95=('global_p95', 'first')
        )
    )
    # compute severity_ratio handling zero P95
    def sev(row):
        gp = row['KPI_global_P95']
        mv = row['max_value_in_segment']
        if pd.isna(gp):
            return None
        if gp == 0:
            return None
        return float(mv / gp)
    seg_agg['severity_ratio'] = seg_agg.apply(sev, axis=1)
    # take up to top 5 segments per KPI sorted by segment_start_timestamp
    seg_agg = seg_agg.sort_values('segment_start_timestamp').head(5)
    segments_list.append(seg_agg)

# Concatenate segments for all KPIs
if segments_list:
    segments_df = pd.concat(segments_list, ignore_index=True)
else:
    segments_df = pd.DataFrame(columns=[
        'kpi_name','segment_start_timestamp','segment_end_timestamp',
        'segment_length_in_minutes','max_value_in_segment','KPI_global_P95','severity_ratio'
    ])

# Sort overall by segment_start_timestamp and limit to 20 rows
segments_top20 = segments_df.sort_values('segment_start_timestamp').head(20).reset_index(drop=True)

# Earliest anomalous segment start across all KPIs and which KPI it belongs to
if not segments_df.empty:
    earliest_idx = segments_df['segment_start_timestamp'].idxmin()
    earliest_overall = segments_df.loc[earliest_idx, 'segment_start_timestamp']
    earliest_kpi = segments_df.loc[earliest_idx, 'kpi_name']
    earliest_summary = pd.DataFrame({
        'earliest_segment_start': [earliest_overall],
        'kpi_name': [earliest_kpi]
    })
else:
    earliest_overall = pd.NaT
    earliest_kpi = None
    earliest_summary = pd.DataFrame(columns=['earliest_segment_start','kpi_name'])

# Return compact outputs
segments_top20, earliest_summary

```
Out[12]:
```
Summary for ts-travel-service (incident window 2024-01-23 06:50:39 → 07:20:39 UTC):

Earliest anomalous segment overall: 2024-01-23 06:53:00 UTC — KPI: workload.

Anomalous segments found (up to top segments, sorted by start time):

- workload
  - 2024-01-23 06:53:00 — 06:53:00 (1 min), max=11.133, global_p95≈9.728, severity≈1.144
  - 2024-01-23 07:02:00 — 07:02:00 (1 min), max=9.772, global_p95≈9.728, severity≈1.004

- cpu
  - 2024-01-23 07:01:00 — 07:02:00 (2 min), max≈3.430, global_p95≈3.203, severity≈1.071

- diskio
  - 2024-01-23 07:02:00 — 07:02:00 (1 min), max≈60,387.81, global_p95≈60,007.77, severity≈1.006
  - 2024-01-23 07:05:00 — 07:05:00 (1 min), max≈60,972.97, global_p95≈60,007.77, severity≈1.016

- socket
  - 2024-01-23 07:06:00 — 07:06:00 (1 min), max≈18.983, global_p95≈18.08, severity≈1.050
  - 2024-01-23 07:15:00 — 07:15:00 (1 min), max≈18.150, global_p95≈18.08, severity≈1.004

- latency-90
  - 2024-01-23 07:09:00 — 07:09:00 (1 min), max≈0.4542, global_p95≈0.45359, severity≈1.0013
  - 2024-01-23 07:17:00 — 07:17:00 (1 min), max≈0.45562, global_p95≈0.45359, severity≈1.0045

- mem
  - 2024-01-23 07:15:00 — 07:15:00 (1 min), max≈272,246,033.07, global_p95≈272,201,550.51, severity≈1.00016
  - 2024-01-23 07:17:00 — 07:17:00 (1 min), max≈272,202,240.00, global_p95≈272,201,550.51, severity≈1.00000

Interpretation:
- The first anomalous event is a workload spike at 06:53 UTC.
- CPU and diskio anomalies occur shortly after (~07:01–07:05), with socket and latency anomalies following (~07:06–07:17).
- Most segments are single-minute spikes; CPU shows a 2-minute anomalous span.
- Severity ratios are modest (mostly ~1.00–1.15), with the largest relative spike on workload at 06:53 (≈1.144).
- Recommended next step: investigate the workload spike at 06:53 and subsequent CPU/disk I/O spikes (07:01–07:05) and correlate with traces and caller activity to find downstream cause.

The original code execution output of IPython Kernel is also provided below for reference:

(    segment_id    kpi_name   segment_start_timestamp     segment_end_timestamp  segment_length_in_minutes  max_value_in_segment   KPI_global_P95  severity_ratio
0            1    workload 2024-01-23 06:53:00+00:00 2024-01-23 06:53:00+00:00                          1             11.133143         9.728423        1.144393
1            1         cpu 2024-01-23 07:01:00+00:00 2024-01-23 07:02:00+00:00                          2              3.429573         3.203012        1.070734
2            1      diskio 2024-01-23 07:02:00+00:00 2024-01-23 07:02:00+00:00                          1          60387.811647     60007.766282        1.006333
3            2    workload 2024-01-23 07:02:00+00:00 2024-01-23 07:02:00+00:00                          1              9.772150         9.728423        1.004495
4            2      diskio 2024-01-23 07:05:00+00:00 2024-01-23 07:05:00+00:00                          1          60972.969086     60007.766282        1.016085
5            1      socket 2024-01-23 07:06:00+00:00 2024-01-23 07:06:00+00:00                          1             18.983333        18.080000        1.049963
6            1  latency-90 2024-01-23 07:09:00+00:00 2024-01-23 07:09:00+00:00                          1              0.454165         0.453590        1.001269
7            1         mem 2024-01-23 07:15:00+00:00 2024-01-23 07:15:00+00:00                          1      272246033.066667 272201550.506667        1.000163
8            2      socket 2024-01-23 07:15:00+00:00 2024-01-23 07:15:00+00:00                          1             18.150000        18.080000        1.003872
9            2  latency-90 2024-01-23 07:17:00+00:00 2024-01-23 07:17:00+00:00                          1              0.455620         0.453590        1.004475
10           2         mem 2024-01-23 07:17:00+00:00 2024-01-23 07:17:00+00:00                          1      272202240.000000 272201550.506667        1.000003,      earliest_segment_start  kpi_name
0 2024-01-23 06:53:00+00:00  workload)```
```

In [ ]:
# In[13]:

import pandas as pd
import numpy as np

# Load trace.csv and parse timestamps as UTC datetimes (Unix seconds)
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Compute global P95 per (cmdb_id, trace_name) using the full series (before filtering)
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Incident window (inclusive)
win_start = pd.to_datetime("2024-01-23 06:50:39", utc=True)
win_end   = pd.to_datetime("2024-01-23 07:20:39", utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= win_start) & (trace_df['timestamp'] <= win_end)].copy()

# Select traces where trace_name contains 'to_ts-travel-service' (caller -> travel)
mask_to_travel = trace_window['trace_name'].str.contains(r'to_ts-travel-service', na=False)
trace_to_travel = trace_window[mask_to_travel].copy()

# Further restrict to early sub-window around travel's first anomaly (inclusive)
sub_start = pd.to_datetime("2024-01-23 06:50:00", utc=True)
sub_end   = pd.to_datetime("2024-01-23 06:56:00", utc=True)
trace_sub = trace_to_travel[(trace_to_travel['timestamp'] >= sub_start) & (trace_to_travel['timestamp'] <= sub_end)].copy()

# Attach global_p95 computed from the full series
trace_sub = trace_sub.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# Mark anomalies per rule: value >= global_p95 OR (global_p95 == 0 and value > 0)
trace_sub['is_anomaly'] = np.where(
    trace_sub['global_p95'] == 0,
    trace_sub['value'] > 0,
    trace_sub['value'] >= trace_sub['global_p95']
)

# Aggregate by caller (cmdb_id)
# anomaly_count, earliest_anomaly_timestamp, latest_anomaly_timestamp (from anomalous rows)
anom_rows = trace_sub[trace_sub['is_anomaly']].copy()
anom_agg = (
    anom_rows
    .groupby('cmdb_id', as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        latest_anomaly_timestamp=('timestamp', 'max')
    )
)

# max_value_in_window per caller (consider all rows in the sub-window)
max_vals = (
    trace_sub
    .groupby('cmdb_id', as_index=False)
    .agg(max_value_in_window=('value', 'max'))
)

# Representative trace_global_p95: choose the global_p95 of the row with the max value per caller
idx_max = trace_sub.groupby('cmdb_id')['value'].idxmax().dropna().astype(int)
rep_gp = trace_sub.loc[idx_max, ['cmdb_id', 'global_p95']].drop_duplicates(subset=['cmdb_id']).rename(columns={'global_p95':'trace_global_p95'})

# Merge aggregates
summary = anom_agg.merge(max_vals, on='cmdb_id', how='right').merge(rep_gp, on='cmdb_id', how='left')

# Fill anomaly_count for callers with 0 anomalies
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

# Compute severity_ratio handling zero or NaN global_p95
def compute_severity(row):
    gp = row['trace_global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp):
        return None
    if gp == 0:
        return None
    return float(mv / gp)

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# Rename cmdb_id -> caller and order columns
summary = summary.rename(columns={'cmdb_id':'caller'})
summary = summary.loc[:, ['caller', 'anomaly_count', 'earliest_anomaly_timestamp',
                          'latest_anomaly_timestamp', 'max_value_in_window',
                          'trace_global_p95', 'severity_ratio']]

# Sort by anomaly_count desc then severity_ratio desc, limit to top 20
summary_top = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Distinct ordered list of callers found (order by earliest timestamp in subwindow)
callers_order = (
    trace_sub.groupby('cmdb_id', as_index=False)['timestamp']
    .min()
    .sort_values('timestamp')
    .rename(columns={'cmdb_id':'caller', 'timestamp':'first_seen_timestamp'})
)
callers_ordered = callers_order['caller'].reset_index(drop=True)

# Return compact results
summary_top, callers_ordered

```
Out[13]:
```
Summary (UTC), caller → travel-service, early sub-window 2024-01-23 06:50:00 — 06:56:00:

- Findings (aggregated by caller):
  - ts-admin-travel-service — anomaly_count = 2; earliest = 2024-01-23 06:54:00; latest = 2024-01-23 06:55:00; max_trace=11.0; trace_global_p95=11.0; severity = 1.00.
  - ts-food-service — anomaly_count = 1; earliest = 2024-01-23 06:55:00; latest = 2024-01-23 06:55:00; max_trace=31.0; trace_global_p95=26.25; severity ≈ 1.18.
  - ts-travel-service (self-calls) — anomaly_count = 0; max_trace=1314.0 (below P95), severity ≈ 0.77.
  - ts-seat-service — anomaly_count = 0; max_trace=186.0 (below P95), severity ≈ 0.77.
  - root — anomaly_count = 0; max_trace=19.0 (below P95), severity ≈ 0.76.
  - ts-preserve-service — anomaly_count = 0; max_trace=3.0 (below P95), severity ≈ 0.50.

- Distinct callers observed (chronological order by first seen): root, ts-admin-travel-service, ts-food-service, ts-preserve-service, ts-seat-service, ts-travel-service.

Interpretation and next step:
- The early anomalous callers to travel are primarily ts-admin-travel-service (two anomalous points, exactly at its P95) and ts-food-service (one stronger exceedance, severity ≈ 1.18). Other callers made calls but did not exceed their trace P95 in this early window.
- Focus immediate investigation on ts-admin-travel-service and ts-food-service call paths into travel around 06:54–06:55 UTC, and correlate with travel-service metrics/logs at the same timestamps to determine whether the issue originates at callers (traffic spike) or impacts the travel-service downstream.

The original code execution output of IPython Kernel is also provided below for reference:

(                    caller  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  trace_global_p95  severity_ratio
0  ts-admin-travel-service              2  2024-01-23 06:54:00+00:00 2024-01-23 06:55:00+00:00            11.000000         11.000000        1.000000
1          ts-food-service              1  2024-01-23 06:55:00+00:00 2024-01-23 06:55:00+00:00            31.000000         26.250000        1.180952
2        ts-travel-service              0                        NaT                       NaT          1314.000000       1695.850000        0.774833
3          ts-seat-service              0                        NaT                       NaT           186.000000        240.700000        0.772746
4                     root              0                        NaT                       NaT            19.000000         24.900000        0.763052
5      ts-preserve-service              0                        NaT                       NaT             3.000000          6.050000        0.495868, 0                       root
1    ts-admin-travel-service
2            ts-food-service
3        ts-preserve-service
4            ts-seat-service
5          ts-travel-service
Name: caller, dtype: object)```
```